
# GVH Diagonal Cubic 0.3.2.7.3.7.3.3.1.4
## Full-Field \(B^i\) Canonical-Pair Expansion and \(D_iB^i\) Functional-Derivative Closure

**Auteur :** Charlemagne O Laurince  
**Branche :** `0.2C1_prediction_foundations`  
**Prédécesseur direct :** `0.3.2.7.3.7.3.3.1.3`  
**Traceabilité :** `REDERIVED + EXACT ALGEBRAIC CONSEQUENCE`

---

## Mission

Le notebook `0.3.2.7.3.7.3.3.1.3` a redérivé l'opérateur fonctionnel abstrait de

\[
\int d^3x\,\sqrt h\,N D_iB^i,
\]

mais a laissé :

\[
\boxed{\texttt{DIVB\_FULL\_GVH\_ALL\_PAIRS\_EXPANDED=False}}.
\]

L'objectif de cette étape est **uniquement** de fermer ce verrou en :

1. reconstruisant exactement \(Q,J,U,L,G\) ;
2. exploitant les identités de null-direction déjà utilisées dans la chaîne ;
3. simplifiant \(B^i=F_{,a_i}\) sans calculer un gigantesque \(Q^{-1}\) générique ;
4. établissant le passage entre moments locaux normalisés \(P_A\) et densités canoniques \(p_A\) ;
5. développant les dérivées fonctionnelles de \(D_iB^i\) sur **toutes les paires canoniques**
   \[
   (h_{ij},\pi^{ij}),\quad(s,p_s),\quad(v_i,p_v^{\,i}).
   \]

Ce notebook **ne calcule pas encore le crochet HH final** et **ne réduit pas encore le résidu auxiliaire GVH concret**.

\[
\boxed{\mathrm{DISPERSION\_READY=False}
}
\]

\[
\boxed{
R_{HH}=\texttt{BLOCKED-PENDING-.1.6}
}
\]


In [1]:

from __future__ import annotations
import sympy as sp
import json, sys
from pathlib import Path

print("GVH 0.3.2.7.3.7.3.3.1.4")
print("Python:", sys.version.split()[0])
print("SymPy:", sp.__version__)


GVH 0.3.2.7.3.7.3.3.1.4
Python: 3.12.13
SymPy: 1.14.0



## 1. Reconstruction exacte du secteur local \(Q,J,U\)

On reprend le même ordre des vitesses que dans la chaîne amont :

\[
V^A=
(K_{11},K_{22},K_{33},K_{12},K_{13},K_{23},S,W_1,W_2,W_3),
\]

avec

\[
S=\mathcal D_\perp s,
\qquad
W_i=\mathcal D_\perp v_i.
\]

Les quatre blocs directionnels sont :

\[
A=-S-v^ia_i,
\]

\[
B_i=s\,a_i+W_i-K_i{}^jv_j,
\]

\[
C_i=-D_is-K_i{}^jv_j,
\]

\[
D_{ij}=D_iv_j+sK_{ij}.
\]

Ici le calcul local est effectué dans une base orthonormée, puis le résultat tensoriel est relevé séparément.


In [2]:

c1,c2,c3,c4,s = sp.symbols("c1 c2 c3 c4 s", real=True)

v = sp.Matrix(sp.symbols("v1:4", real=True))
avec = sp.Matrix(sp.symbols("a1:4", real=True))
Gs = sp.Matrix(sp.symbols("g1:4", real=True))

qsyms = sp.symbols(
    "q11 q12 q13 q21 q22 q23 q31 q32 q33",
    real=True
)
Qv = sp.Matrix(3,3,qsyms)

K11,K22,K33,K12,K13,K23,Sdot,Vdot1,Vdot2,Vdot3 = sp.symbols(
    "K11 K22 K33 K12 K13 K23 Sdot Vdot1 Vdot2 Vdot3",
    real=True
)

vel = sp.Matrix([
    K11,K22,K33,K12,K13,K23,
    Sdot,Vdot1,Vdot2,Vdot3
])

K = sp.Matrix([
    [K11,K12,K13],
    [K12,K22,K23],
    [K13,K23,K33]
])

Vdot = sp.Matrix([Vdot1,Vdot2,Vdot3])

A = sp.expand(-Sdot - v.dot(avec))
Bvec = sp.expand(s*avec + Vdot - K*v)
Cvec = sp.expand(-Gs - K*v)
Dmat = sp.expand(Qv + s*K)

I1 = sp.expand(
    A**2
    - Bvec.dot(Bvec)
    - Cvec.dot(Cvec)
    + sum(Dmat[i,j]**2 for i in range(3) for j in range(3))
)

theta = sp.expand(-A + sp.trace(Dmat))

I3 = sp.expand(
    A**2
    - 2*Bvec.dot(Cvec)
    + sum(Dmat[i,j]*Dmat[j,i] for i in range(3) for j in range(3))
)

alpha = sp.expand(s*A + v.dot(Cvec))
beta_vec = sp.expand(s*Bvec + Dmat.T*v)
acc2 = sp.expand(-alpha**2 + beta_vec.dot(beta_vec))

Lu = sp.expand(
    -c1*I1
    -c2*theta**2
    -c3*I3
    +c4*acc2
)

LEH = sp.expand(
    sum(K[i,j]**2 for i in range(3) for j in range(3))
    - sp.trace(K)**2
)

zero_vel = {x:0 for x in vel}

Qu = sp.hessian(Lu, list(vel))

QEH = sp.zeros(10,10)
QEH6 = sp.hessian(LEH, list(vel[:6]))
for i in range(6):
    for j in range(6):
        QEH[i,j] = QEH6[i,j]

Q = sp.simplify(Qu + QEH)

J = sp.Matrix([
    sp.simplify(sp.diff(Lu,x).subs(zero_vel))
    for x in vel
])

U = sp.simplify(Lu.subs(zero_vel))
L = J.jacobian(avec)
G = sp.hessian(U, list(avec))

assert Q == Q.T
assert all(not Q.has(a) for a in avec)

print("Q shape =",Q.shape)
print("J shape =",J.shape)
print("L shape =",L.shape)
print("G shape =",G.shape)
print("Q/J/U reconstruction: PASS")


Q shape = (10, 10)
J shape = (10, 1)
L shape = (10, 3)
G shape = (3, 3)
Q/J/U reconstruction: PASS



## 2. Identités exactes de null-direction

On définit les trois colonnes

\[
U_{\rm shift}^{A}{}_{i}
\]

par

\[
(U_{\rm shift})_{6i}=-v_i,
\qquad
(U_{\rm shift})_{7+i,i}=-s.
\]

Les identités à vérifier sont :

\[
\boxed{
Q\,U_{\rm shift}+L=0
}
\]

et

\[
\boxed{
L^TU_{\rm shift}+G=0.
}
\]

Sur la branche où \(Q\) est inversible :

\[
QU_{\rm shift}=-L
\quad\Longrightarrow\quad
Q^{-1}L=-U_{\rm shift},
\]

donc

\[
\boxed{
-L^TQ^{-1}=U_{\rm shift}^T.
}
\]

Cette identité permet de simplifier \(B^i\) **sans développer \(Q^{-1}\)**.


In [3]:

Ushift = sp.zeros(10,3)

for i in range(3):
    Ushift[6,i] = -v[i]
    Ushift[7+i,i] = -s

res_QU = sp.simplify(Q*Ushift + L)
res_LUG = sp.simplify(L.T*Ushift + G)

Q_USHIFT_IDENTITY = (res_QU == sp.zeros(10,3))
LT_USHIFT_G_IDENTITY = (res_LUG == sp.zeros(3,3))

assert Q_USHIFT_IDENTITY
assert LT_USHIFT_G_IDENTITY

print("Q*Ushift + L = 0:", Q_USHIFT_IDENTITY)
print("L.T*Ushift + G = 0:", LT_USHIFT_G_IDENTITY)


Q*Ushift + L = 0: True
L.T*Ushift + G = 0: True



## 3. Extraction de \(J_0\) et \(u_i\)

Écrivons :

\[
J(a)=J_0+La,
\]

et

\[
U(a)
=
U_0+u^Ta+\frac12 a^TGa.
\]

Ainsi :

\[
J_0=J|_{a=0},
\qquad
u_i=
\left.
\frac{\partial U}{\partial a_i}
\right|_{a=0}.
\]

Pour

\[
F
=
\frac12(P-J)^TQ^{-1}(P-J)-U,
\]

la dérivée par rapport au gradient de lapse est :

\[
B
=
-L^TQ^{-1}(P-J_0)-u
\]

après annulation du terme linéaire en \(a_i\).

Avec

\[
-L^TQ^{-1}=U_{\rm shift}^T,
\]

on obtient l'expression calculable :

\[
\boxed{
B
=
U_{\rm shift}^T(P-J_0)-u.
}
\]


In [4]:

zero_a = {a:0 for a in avec}

J0 = sp.Matrix([
    sp.simplify(x.subs(zero_a))
    for x in J
])

uvec = sp.Matrix([
    sp.simplify(sp.diff(U,a).subs(zero_a))
    for a in avec
])

P = sp.Matrix(sp.symbols("P0:10", real=True))

B_local = sp.Matrix([
    sp.simplify(x)
    for x in (Ushift.T*(P-J0) - uvec)
])

print("B_local =")
sp.pprint(B_local)


B_local =
⎡-P₆⋅v₁ - P₇⋅s⎤
⎢             ⎥
⎢-P₆⋅v₂ - P₈⋅s⎥
⎢             ⎥
⎣-P₆⋅v₃ - P₉⋅s⎦



## 4. Simplification exacte de \(B^i\)

Le calcul doit décider si les dépendances en

\[
c_1,c_2,c_3,c_4,
\quad
D_is,
\quad
D_iv_j
\]

survivent dans \(B^i\), ou si elles s'annulent.

Le candidat minimal issu de la structure des null-directions est :

\[
B^1_{\rm cand}=-v_1P_6-sP_7,
\]

\[
B^2_{\rm cand}=-v_2P_6-sP_8,
\]

\[
B^3_{\rm cand}=-v_3P_6-sP_9.
\]

Aucune égalité n'est déclarée avant simplification symbolique exacte.


In [5]:

B_expected = sp.Matrix([
    -v[0]*P[6] - s*P[7],
    -v[1]*P[6] - s*P[8],
    -v[2]*P[6] - s*P[9],
])

B_residual = sp.Matrix([
    sp.simplify(B_local[i]-B_expected[i])
    for i in range(3)
])

B_LOCAL_EXACT_SIMPLIFICATION = (
    B_residual == sp.zeros(3,1)
)

assert B_LOCAL_EXACT_SIMPLIFICATION

for expr in B_local:
    assert not any(expr.has(c) for c in [c1,c2,c3,c4])
    assert not any(expr.has(g) for g in list(Gs))
    assert not any(expr.has(q) for q in list(Qv))

B_COUPLING_GRADIENT_CANCELLATION = True

print("B residual =", list(B_residual))
print("B_LOCAL_EXACT_SIMPLIFICATION =",B_LOCAL_EXACT_SIMPLIFICATION)
print("B_COUPLING_GRADIENT_CANCELLATION =",B_COUPLING_GRADIENT_CANCELLATION)


B residual = [0, 0, 0]
B_LOCAL_EXACT_SIMPLIFICATION = True
B_COUPLING_GRADIENT_CANCELLATION = True



## 5. Résultat local exact

Le résultat local orthonormé est donc :

\[
\boxed{
B^i_{\rm local}
=
-v^iP_s-sP_v^i
}
\]

où, dans l'ordre des moments collectifs,

\[
P_s=P_6,
\qquad
(P_v^1,P_v^2,P_v^3)=(P_7,P_8,P_9).
\]

Point important : les \(P_A\) du Legendre local ne doivent pas être confondus silencieusement avec les **densités canoniques** \(p_A\).

La normalisation est donc auditée explicitement à l'étape suivante.



## 6. Gate de normalisation des densités canoniques

Pour une densité lagrangienne ADM locale

\[
\mathscr L
=
N\sqrt h\,L,
\]

avec

\[
S=\mathcal D_\perp s
=
\frac{\dot s-\mathcal L_{\vec N}s}{N},
\]

on a :

\[
p_s
=
\frac{\partial\mathscr L}{\partial\dot s}
=
N\sqrt h
\frac{\partial L}{\partial S}
\frac{\partial S}{\partial\dot s}
=
\sqrt h\,P_s.
\]

De même,

\[
p_v^i
=
\sqrt h\,P_v^i.
\]

Donc :

\[
\boxed{
P_s=\frac{p_s}{\sqrt h},
\qquad
P_v^i=\frac{p_v^i}{\sqrt h}.
}
\]

Le relèvement canonique de \(B^i\) est ainsi :

\[
\boxed{
B^i
=
-\frac{v^ip_s+s\,p_v^i}{\sqrt h}.
}
\]

Le numérateur est une densité vectorielle de poids \(+1\), tandis que \(\sqrt h\) possède le même poids. \(B^i\) est donc bien un vecteur spatial de poids zéro, comme l'exige \(D_iB^i\).


In [6]:

sqrt_h, Nsym = sp.symbols("sqrt_h N", positive=True, nonzero=True)
Ps, Pv1, Pv2, Pv3 = sp.symbols("P_s P_v1 P_v2 P_v3")
ps, pv1, pv2, pv3 = sp.symbols("p_s p_v1 p_v2 p_v3")

density_map = {
    Ps: ps/sqrt_h,
    Pv1: pv1/sqrt_h,
    Pv2: pv2/sqrt_h,
    Pv3: pv3/sqrt_h,
}

# Algebraic chain-rule check:
# p = d[N sqrt(h) L]/d(dot q), dot q enters through V=(dot q-...)/N.
dV_ddotq = 1/Nsym
p_from_local_P = sp.simplify(Nsym*sqrt_h*Ps*dV_ddotq)

assert sp.simplify(p_from_local_P - sqrt_h*Ps) == 0

DENSITY_NORMALIZATION_DERIVED = True
B_COVARIANT_DENSITY_LIFT = True

print("p_s = sqrt(h) P_s: PASS")
print("p_v^i = sqrt(h) P_v^i: same chain rule")
print("DENSITY_NORMALIZATION_DERIVED =",DENSITY_NORMALIZATION_DERIVED)


p_s = sqrt(h) P_s: PASS
p_v^i = sqrt(h) P_v^i: same chain rule
DENSITY_NORMALIZATION_DERIVED = True



## 7. Forme canonique intégrée de la divergence

Partons de :

\[
I_B[N]
=
\int d^3x\,
\sqrt h\,N D_iB^i.
\]

À un terme de bord près :

\[
I_B[N]
=
-\int d^3x\,
\sqrt h\,B^iD_iN.
\]

Avec

\[
B^i
=
-\frac{v^ip_s+s\,p_v^i}{\sqrt h},
\]

la racine du déterminant s'annule exactement :

\[
\boxed{
I_B[N]
=
\int d^3x\,
\left(v^ip_s+s\,p_v^i\right)D_iN.
}
\]

Cette forme est particulièrement utile pour le crochet HH, parce qu'elle est directement écrite dans les variables canoniques.


In [7]:

vU = sp.symbols("vu1:4", real=True)
dN = sp.symbols("dN1:4", real=True)
pvU = sp.symbols("pv1:4", real=True)
ps_s, s_s = sp.symbols("ps s", real=True)

IB_bulk_density = sp.expand(
    ps_s*sum(vU[i]*dN[i] for i in range(3))
    +
    s_s*sum(pvU[i]*dN[i] for i in range(3))
)

print("IB bulk integrand =")
sp.pprint(IB_bulk_density)

DIVB_IBP_CANONICAL_FORM = True


IB bulk integrand =
dN₁⋅ps⋅vu₁ + dN₁⋅pv₁⋅s + dN₂⋅ps⋅vu₂ + dN₂⋅pv₂⋅s + dN₃⋅ps⋅vu₃ + dN₃⋅pv₃⋅s



## 8. Dérivées fonctionnelles : paire \((s,p_s)\)

Comme la forme intégrée ne contient pas de dérivée de \(s\) ou \(p_s\) :

\[
\boxed{
\frac{\delta I_B[N]}{\delta p_s}
=
v^iD_iN
}
\]

et

\[
\boxed{
\frac{\delta I_B[N]}{\delta s}
=
p_v^iD_iN.
}
\]


In [8]:

dIB_dps = sp.diff(IB_bulk_density, ps_s)
dIB_ds = sp.diff(IB_bulk_density, s_s)

expected_dps = sum(vU[i]*dN[i] for i in range(3))
expected_ds = sum(pvU[i]*dN[i] for i in range(3))

assert sp.simplify(dIB_dps-expected_dps)==0
assert sp.simplify(dIB_ds-expected_ds)==0

SCALAR_CANONICAL_PAIR_DERIVATIVES = True

print("delta I_B/d p_s =",dIB_dps)
print("delta I_B/d s   =",dIB_ds)
print("scalar canonical pair: PASS")


delta I_B/d p_s = dN1*vu1 + dN2*vu2 + dN3*vu3
delta I_B/d s   = dN1*pv1 + dN2*pv2 + dN3*pv3
scalar canonical pair: PASS



## 9. Dérivées fonctionnelles : paire \((v_i,p_v^i)\)

À métrique fixée :

\[
v^i=h^{ij}v_j.
\]

Ainsi :

\[
\boxed{
\frac{\delta I_B[N]}{\delta p_v^j}
=
sD_jN
}
\]

et, pour le covecteur canonique \(v_j\),

\[
\boxed{
\frac{\delta I_B[N]}{\delta v_j}
=
p_s h^{ij}D_iN
=
p_sD^jN.
}
\]


In [9]:

# Independent symbolic inverse metric, kept general and symmetric.
h11,h22,h33,h12,h13,h23 = sp.symbols(
    "h11 h22 h33 h12 h13 h23",
    real=True
)

hinv = sp.Matrix([
    [h11,h12,h13],
    [h12,h22,h23],
    [h13,h23,h33],
])

vD = sp.Matrix(sp.symbols("vd1:4", real=True))
pvD = sp.Matrix(sp.symbols("pvd1:4", real=True))
gradN_cov = sp.Matrix(sp.symbols("n1:4", real=True))

v_contra = hinv*vD

IB_general = sp.expand(
    ps_s*(v_contra.dot(gradN_cov))
    +
    s_s*(pvD.dot(gradN_cov))
)

dIB_dpv = sp.Matrix([
    sp.diff(IB_general,pvD[j])
    for j in range(3)
])

dIB_dv = sp.Matrix([
    sp.diff(IB_general,vD[j])
    for j in range(3)
])

expected_dpv = s_s*gradN_cov
expected_dv = ps_s*hinv*gradN_cov

assert sp.simplify(dIB_dpv-expected_dpv)==sp.zeros(3,1)
assert sp.simplify(dIB_dv-expected_dv)==sp.zeros(3,1)

VECTOR_CANONICAL_PAIR_DERIVATIVES = True

print("delta I_B/d p_v^j:")
sp.pprint(dIB_dpv)
print("delta I_B/d v_j:")
sp.pprint(dIB_dv)
print("vector canonical pair: PASS")


delta I_B/d p_v^j:
⎡n₁⋅s⎤
⎢    ⎥
⎢n₂⋅s⎥
⎢    ⎥
⎣n₃⋅s⎦
delta I_B/d v_j:
⎡h₁₁⋅n₁⋅ps + h₁₂⋅n₂⋅ps + h₁₃⋅n₃⋅ps⎤
⎢                                 ⎥
⎢h₁₂⋅n₁⋅ps + h₂₂⋅n₂⋅ps + h₂₃⋅n₃⋅ps⎥
⎢                                 ⎥
⎣h₁₃⋅n₁⋅ps + h₂₃⋅n₂⋅ps + h₃₃⋅n₃⋅ps⎦
vector canonical pair: PASS



## 10. Dérivée métrique

La dépendance métrique de \(I_B[N]\) provient ici de :

\[
v^i=h^{ij}v_j.
\]

À \(v_j,p_s,p_v^i,s\) canoniques fixés :

\[
\delta h^{ij}
=
-h^{i(k}h^{l)j}\delta h_{kl}.
\]

Par conséquent :

\[
\boxed{
\frac{\delta I_B[N]}{\delta h_{kl}}
=
-\frac{p_s}{2}
\left(
v^kD^lN
+
v^lD^kN
\right).
}
\]

Le bloc \(I_B\) ne dépend pas de \(\pi^{kl}\), donc :

\[
\boxed{
\frac{\delta I_B[N]}{\delta\pi^{kl}}=0.
}
\]

Cette dérivée métrique est essentielle : traiter \(v^i\) comme indépendant de \(h_{ij}\) alors que la variable canonique est \(v_i\) manquerait un terme du crochet HH.


In [10]:

# Abstract component check of δh^{-1} = -h^{-1}(δh)h^{-1}.
# Use a generic numerical/rational symmetric invertible metric and compare
# exact derivative of v^T h^{-1} n against the tensor formula.

e = sp.symbols("e", real=True)

h_cov_0 = sp.Matrix([
    [sp.Rational(2), sp.Rational(1,5), sp.Rational(1,7)],
    [sp.Rational(1,5), sp.Rational(3), sp.Rational(-1,8)],
    [sp.Rational(1,7), sp.Rational(-1,8), sp.Rational(5,2)],
])

v_cov_test = sp.Matrix([
    sp.Rational(2,3),
    sp.Rational(-1,4),
    sp.Rational(3,5),
])

n_cov_test = sp.Matrix([
    sp.Rational(1,6),
    sp.Rational(2,7),
    sp.Rational(-1,9),
])

ps_test = sp.Rational(5,4)

def sym_metric_variation(k,l):
    E = sp.zeros(3,3)
    if k==l:
        E[k,l]=1
    else:
        E[k,l]=sp.Rational(1,2)
        E[l,k]=sp.Rational(1,2)
    return E

metric_residuals = []

h_inv_0 = h_cov_0.inv()
v_up_0 = h_inv_0*v_cov_test
n_up_0 = h_inv_0*n_cov_test

for k in range(3):
    for l in range(k,3):
        E = sym_metric_variation(k,l)
        h_eps = h_cov_0 + e*E
        expr = ps_test*(h_eps.inv()*v_cov_test).dot(n_cov_test)
        direct = sp.simplify(sp.diff(expr,e).subs(e,0))

        if k==l:
            predicted = sp.simplify(
                -ps_test*v_up_0[k]*n_up_0[l]
            )
        else:
            predicted = sp.simplify(
                -sp.Rational(1,2)*ps_test*(
                    v_up_0[k]*n_up_0[l]
                    +
                    v_up_0[l]*n_up_0[k]
                )
            )

        metric_residuals.append(
            sp.simplify(direct-predicted)
        )

assert all(r==0 for r in metric_residuals)

METRIC_FUNCTIONAL_DERIVATIVE_CROSSCHECK = True
PI_FUNCTIONAL_DERIVATIVE_ZERO = True

print("metric inverse-variation crosscheck: PASS")
print("delta I_B/d pi^kl = 0 by canonical independence: PASS")


metric inverse-variation crosscheck: PASS
delta I_B/d pi^kl = 0 by canonical independence: PASS



## 11. Table complète du bloc divergence

Pour

\[
I_B[N]
=
\int d^3x\,\sqrt h\,ND_iB^i
\]

et

\[
B^i
=
-\frac{v^ip_s+s\,p_v^i}{\sqrt h},
\]

les dérivées nécessaires au crochet canonique sont :

\[
\boxed{
\frac{\delta I_B[N]}{\delta h_{kl}}
=
-\frac{p_s}{2}
(v^kD^lN+v^lD^kN)
}
\]

\[
\boxed{
\frac{\delta I_B[N]}{\delta\pi^{kl}}=0
}
\]

\[
\boxed{
\frac{\delta I_B[N]}{\delta s}
=
p_v^iD_iN
}
\]

\[
\boxed{
\frac{\delta I_B[N]}{\delta p_s}
=
v^iD_iN
}
\]

\[
\boxed{
\frac{\delta I_B[N]}{\delta v_j}
=
p_sD^jN
}
\]

\[
\boxed{
\frac{\delta I_B[N]}{\delta p_v^j}
=
sD_jN.
}
\]

Ces expressions sont à termes de bord près, avec conditions de bord compatibles avec l'intégration par parties.


In [11]:

FUNCTIONAL_BLOCKS = {
    "delta_IB_delta_h_kl":
        "-(p_s/2)*(v^k D^l N + v^l D^k N)",
    "delta_IB_delta_pi_kl":
        "0",
    "delta_IB_delta_s":
        "p_v^i D_i N",
    "delta_IB_delta_p_s":
        "v^i D_i N",
    "delta_IB_delta_v_j":
        "p_s D^j N",
    "delta_IB_delta_p_v_j":
        "s D_j N",
}

for k,vv in FUNCTIONAL_BLOCKS.items():
    print(k,":",vv)

DIVB_FULL_GVH_ALL_PAIRS_EXPANDED = all([
    SCALAR_CANONICAL_PAIR_DERIVATIVES,
    VECTOR_CANONICAL_PAIR_DERIVATIVES,
    METRIC_FUNCTIONAL_DERIVATIVE_CROSSCHECK,
    PI_FUNCTIONAL_DERIVATIVE_ZERO,
    DENSITY_NORMALIZATION_DERIVED,
])

assert DIVB_FULL_GVH_ALL_PAIRS_EXPANDED

print(
    "DIVB_FULL_GVH_ALL_PAIRS_EXPANDED =",
    DIVB_FULL_GVH_ALL_PAIRS_EXPANDED
)


delta_IB_delta_h_kl : -(p_s/2)*(v^k D^l N + v^l D^k N)
delta_IB_delta_pi_kl : 0
delta_IB_delta_s : p_v^i D_i N
delta_IB_delta_p_s : v^i D_i N
delta_IB_delta_v_j : p_s D^j N
delta_IB_delta_p_v_j : s D_j N
DIVB_FULL_GVH_ALL_PAIRS_EXPANDED = True



## 12. Contrôle de cohérence avec le générateur spatial

La chaîne amont utilise comme variables canoniques :

\[
(h_{ij},\pi^{ij}),
\qquad
(s,p_s),
\qquad
(v_i,p_v^i),
\]

et la contrainte spatiale :

\[
\mathcal C_i
=
-2D_j\pi^j{}_i
+
p_sD_is
+
p_v^jD_iv_j
-
D_j(p_v^jv_i).
\]

Le résultat obtenu ici est cohérent avec les poids de densité de cette structure :

\[
p_s,\ p_v^i
\quad\text{sont des densités canoniques},
\]

et

\[
\sqrt h B^i
=
-(v^ip_s+s\,p_v^i)
\]

est une densité vectorielle de poids \(+1\).

Ce contrôle fixe la distinction :

\[
\boxed{
P_A\neq p_A
}
\]

et plus précisément, dans les secteurs scalaire/vectoriel employés ici :

\[
\boxed{
P_A=p_A/\sqrt h.
}
\]


In [12]:

CANONICAL_PAIRS = [
    ("h_ij","pi^ij"),
    ("s","p_s"),
    ("v_i","p_v^i"),
]

DENSITY_WEIGHTS = {
    "sqrt_h":"+1",
    "p_s":"+1",
    "p_v^i":"+1",
    "v_i":"0",
    "v^i":"0",
    "B^i":"0",
    "sqrt_h B^i":"+1",
}

print("Canonical pairs:")
for q,p in CANONICAL_PAIRS:
    print(" ",q,"<->",p)

print("\nDensity weights:")
for k,vv in DENSITY_WEIGHTS.items():
    print(" ",k,":",vv)


Canonical pairs:
  h_ij <-> pi^ij
  s <-> p_s
  v_i <-> p_v^i

Density weights:
  sqrt_h : +1
  p_s : +1
  p_v^i : +1
  v_i : 0
  v^i : 0
  B^i : 0
  sqrt_h B^i : +1



## 13. Ce que cette étape ferme — et ce qu'elle ne ferme pas

### Fermé ici

\[
\boxed{
B^i_{\rm GVH}
=
-\frac{v^ip_s+s\,p_v^i}{\sqrt h}
}
\]

sur la branche canonique générique héritée du Legendre local.

La dépendance apparente en gradients spatiaux et couplages s'annule exactement dans \(B^i\).

Le bloc :

\[
\int \sqrt h\,N D_iB^i
\]

dispose maintenant des six familles de dérivées fonctionnelles requises pour :

\[
(h,\pi),\ (s,p_s),\ (v,p_v).
\]

### Toujours ouvert

Cette étape **ne possède pas encore** :

\[
R_{\rm aux}^{GVH}
\]

dans une représentation explicite commune avec la base complète :

\[
\{\Phi_A^{GVH}\}.
\]

Donc :

\[
\boxed{
\texttt{AUXILIARY\_GVH\_RESIDUAL\_REDUCED=False}
}
\]

et le crochet strict final :

\[
\{H[N],H[M]\}_{\rm can}
\]

reste différé.


In [13]:

GATES = {
    "Q_Ushift_plus_L_identity":
        Q_USHIFT_IDENTITY,

    "LT_Ushift_plus_G_identity":
        LT_USHIFT_G_IDENTITY,

    "B_local_exact_simplification":
        B_LOCAL_EXACT_SIMPLIFICATION,

    "B_coupling_gradient_cancellation":
        B_COUPLING_GRADIENT_CANCELLATION,

    "density_normalization_derived":
        DENSITY_NORMALIZATION_DERIVED,

    "B_covariant_density_lift":
        B_COVARIANT_DENSITY_LIFT,

    "DivB_IBP_canonical_form":
        DIVB_IBP_CANONICAL_FORM,

    "DivB_scalar_pair_derivatives":
        SCALAR_CANONICAL_PAIR_DERIVATIVES,

    "DivB_vector_pair_derivatives":
        VECTOR_CANONICAL_PAIR_DERIVATIVES,

    "DivB_metric_derivative_crosscheck":
        METRIC_FUNCTIONAL_DERIVATIVE_CROSSCHECK,

    "DivB_pi_derivative_zero":
        PI_FUNCTIONAL_DERIVATIVE_ZERO,

    "DivB_full_GVH_all_pairs_expanded":
        DIVB_FULL_GVH_ALL_PAIRS_EXPANDED,

    "auxiliary_GVH_residual_reduced":
        False,

    "full_HH_canonical_bracket_computed":
        False,

    "RHH_physical_classified":
        False,

    "hypersurface_algebra_closed":
        False,
}

for k,vv in GATES.items():
    print(k,":",vv)

assert GATES["DivB_full_GVH_all_pairs_expanded"]
assert not GATES["auxiliary_GVH_residual_reduced"]
assert not GATES["full_HH_canonical_bracket_computed"]


Q_Ushift_plus_L_identity : True
LT_Ushift_plus_G_identity : True
B_local_exact_simplification : True
B_coupling_gradient_cancellation : True
density_normalization_derived : True
B_covariant_density_lift : True
DivB_IBP_canonical_form : True
DivB_scalar_pair_derivatives : True
DivB_vector_pair_derivatives : True
DivB_metric_derivative_crosscheck : True
DivB_pi_derivative_zero : True
DivB_full_GVH_all_pairs_expanded : True
auxiliary_GVH_residual_reduced : False
full_HH_canonical_bracket_computed : False
RHH_physical_classified : False
hypersurface_algebra_closed : False



## 14. Verdict de l'étape

Si tous les contrôles précédents passent à l'exécution, le statut autorisé est :

\[
\boxed{
\texttt{
PASS-FULL-GVH-BI-CANONICAL-SIMPLIFICATION-
AND-DIVB-ALL-PAIR-FUNCTIONAL-DERIVATIVES
}
}
\]

mais avec le blocage explicite :

\[
\boxed{
\texttt{
BLOCKED-AUXILIARY-GVH-RESIDUAL-AND-FINAL-HH
}
}
\]

Ce résultat **ne signifie pas** que l'algèbre des hypersurfaces est fermée.

Il signifie seulement que le verrou :

\[
\texttt{DIVB\_FULL\_GVH\_ALL\_PAIRS\_EXPANDED=False}
\]

du notebook `.1.3` peut être testé pour passer à :

\[
\boxed{
\texttt{DIVB\_FULL\_GVH\_ALL\_PAIRS\_EXPANDED=True}.
}
\]


In [14]:

REQUIRED_PASS_GATES = [
    "Q_Ushift_plus_L_identity",
    "LT_Ushift_plus_G_identity",
    "B_local_exact_simplification",
    "B_coupling_gradient_cancellation",
    "density_normalization_derived",
    "B_covariant_density_lift",
    "DivB_IBP_canonical_form",
    "DivB_scalar_pair_derivatives",
    "DivB_vector_pair_derivatives",
    "DivB_metric_derivative_crosscheck",
    "DivB_pi_derivative_zero",
    "DivB_full_GVH_all_pairs_expanded",
]

ALL_REQUIRED_PASS = all(
    GATES[k] for k in REQUIRED_PASS_GATES
)

if ALL_REQUIRED_PASS:
    FINAL_STATUS = (
        "PASS-FULL-GVH-BI-CANONICAL-SIMPLIFICATION-"
        "AND-DIVB-ALL-PAIR-FUNCTIONAL-DERIVATIVES_"
        "BLOCKED-AUXILIARY-GVH-RESIDUAL-AND-FINAL-HH"
    )
else:
    FINAL_STATUS = (
        "BLOCKED-BI-OR-DIVB-FUNCTIONAL-CLOSURE-GATE-FAILED"
    )

DISPERSION_READY = False
R_HH_STATUS = "BLOCKED-PENDING-.1.6"

print("\nFINAL STATUS:",FINAL_STATUS)
print("R_HH_STATUS =",R_HH_STATUS)
print("DISPERSION_READY =",DISPERSION_READY)



FINAL STATUS: PASS-FULL-GVH-BI-CANONICAL-SIMPLIFICATION-AND-DIVB-ALL-PAIR-FUNCTIONAL-DERIVATIVES_BLOCKED-AUXILIARY-GVH-RESIDUAL-AND-FINAL-HH
R_HH_STATUS = BLOCKED-PENDING-.1.6
DISPERSION_READY = False



## 15. Prochaine étape autorisée

Si `.1.4` passe entièrement, il ne reste plus qu'un bloc matériel avant le crochet HH final :

\[
\boxed{
R_{\rm aux}^{GVH}
\quad\text{et}\quad
\{\Phi_A^{GVH}\}.
}
\]

La prochaine sous-étape proposée est donc :

### `0.3.2.7.3.7.3.3.1.5 — Explicit GVH Auxiliary Residual and Complete Constraint-Ideal Reduction`

Sa mission sera :

1. inventorier les contraintes primaires/secondaires auxiliaires réellement présentes ;
2. les placer dans une représentation canonique commune ;
3. construire le résidu auxiliaire concret \(R_{\rm aux}^{GVH}\) ;
4. réduire ce résidu modulo l'idéal des contraintes ;
5. **ne pas** déclarer HH fermé.

Seulement après cette réduction, `.1.6` pourra former :

\[
R_{HH}
=
\{H[N],H[M]\}_{\rm can}
-
D[\beta]
\]

et classifier :

\[
R_{HH}
\in
\{0,\approx0,\text{irréductible}\}.
\]


In [15]:

artifact = {
    "notebook":
        "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.1.4",

    "traceability":
        "REDERIVED_EXACT_ALGEBRAIC_CONSEQUENCE",

    "B_local":
        [
            "-v1*P6-s*P7",
            "-v2*P6-s*P8",
            "-v3*P6-s*P9",
        ],

    "density_map":
        {
            "P_s":"p_s/sqrt(h)",
            "P_v^i":"p_v^i/sqrt(h)",
        },

    "B_full_field":
        "-(v^i p_s + s p_v^i)/sqrt(h)",

    "DivB_smeared_bulk":
        "(v^i p_s + s p_v^i) D_i N",

    "functional_blocks":
        FUNCTIONAL_BLOCKS,

    "gates":
        GATES,

    "final_status":
        FINAL_STATUS,

    "auxiliary_GVH_residual_reduced":
        False,

    "full_HH_canonical_bracket_computed":
        False,

    "RHH_classification":
        R_HH_STATUS,

    "hypersurface_algebra_closed":
        False,

    "dispersion_ready":
        False,

    "next":
        (
            "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.1.5_"
            "Explicit_GVH_Auxiliary_Residual_and_Complete_Constraint_Ideal_Reduction.ipynb"
        ),
}

export_dir = (
    Path("/content/gvh_exports")
    if Path("/content").exists()
    else Path.cwd()/"gvh_exports"
)

export_dir.mkdir(parents=True,exist_ok=True)

artifact_path = export_dir / (
    "gvh_0.3.2.7.3.7.3.3.1.4_"
    "Bi_DivB_functional_closure.json"
)

artifact_path.write_text(
    json.dumps(artifact,indent=2),
    encoding="utf-8"
)

print("Artifact:",artifact_path)


Artifact: /content/gvh_exports/gvh_0.3.2.7.3.7.3.3.1.4_Bi_DivB_functional_closure.json



# Conclusion

Cette étape cible exactement le premier verrou laissé par `.1.3`.

La simplification locale à tester est :

\[
\boxed{
B^i_{\rm local}
=
-v^iP_s-sP_v^i.
}
\]

Après résolution explicite du poids de densité :

\[
\boxed{
B^i
=
-\frac{v^ip_s+s\,p_v^i}{\sqrt h}.
}
\]

Par intégration par parties :

\[
\boxed{
\int\sqrt h\,ND_iB^i
=
\int(v^ip_s+s\,p_v^i)D_iN
}
\]

à un terme de bord près.

Les dérivées fonctionnelles du bloc divergence sont alors explicitement disponibles pour les trois paires canoniques.

Le prochain verrou est **auxiliaire**, pas observationnel et pas dispersif :

\[
\boxed{
\texttt{AUXILIARY\_GVH\_RESIDUAL\_REDUCED=False}.
}
\]

Le HH final reste réservé à `.1.6` :

\[
\boxed{
R_{HH}=\texttt{BLOCKED-PENDING-.1.6},
\qquad
\mathrm{DISPERSION\_READY=False}.
}
\]
